# Coastal — consensus workflow (post-audit)

One clean, end-to-end path through the pipeline **as it stands after the audit cleanup**. This is
the *settled* workflow; the many tried-and-removed alternatives (ABM tracker, HMM morphology,
learned appearance / collective / persistence / exclusion / breadcrumb cost terms) are gone from
the code and recorded in [`../docs/DEAD_ENDS.md`](../docs/DEAD_ENDS.md).

**Pipeline:** cecelia loads the raw array → flow-metric UNet (no labels) → 3D+T instance labels →
Kalman + LAP tracking (Mahalanobis gate + `w_flow` + `w_color`) → scoring.

**Cecelia seam (data-loading direction):** cecelia does I/O; coastal takes a normalised array in and
returns a label volume out (see [`../docs/DATA.md`](../docs/DATA.md)). This notebook *consumes*
cecelia's IO helpers; it does **not** make cecelia depend on coastal (that direction is deferred —
see [`../docs/JULIA_PORT.md`](../docs/JULIA_PORT.md)).

**Kernel:** run in **Python (coastal)** (the pixi env — `pixi run kernel`; see `../docs/DATA.md`).

In [ ]:
import os
import numpy as np
import coastal

# cecelia IO helpers (installed editable in the pixi env). See docs/DATA.md.
import cecelia.utils.zarr_utils as zarr_utils
import cecelia.utils.ome_xml_utils as ome_xml_utils
from cecelia.utils.dim_utils import DimUtils

# NOTE: coastal's train/segment entry points default device='cuda' (no auto-fallback yet — a known
# item). On this GPU box 'cuda' is correct; set 'cpu' if running without a GPU.
DEVICE = 'cuda'
print('coastal', coastal.__version__)

## 1. Load a movie (cecelia — the array-boundary seam)

cecelia reads the OME-ZARR and the physical pixel sizes; coastal only ever sees the numpy/dask
array and `pix_res`. Path is an env var so this runs on any machine.

In [ ]:
IM_PATH = os.environ.get('COASTAL_DEMO_IMAGE', '/path/to/your/confetti_movie.zarr')

im, _ = zarr_utils.open_as_zarr(IM_PATH, as_dask=True)            # list of pyramid levels
dim_utils = DimUtils(ome_xml_utils.parse_meta(IM_PATH), use_channel_axis=True)
dim_utils.calc_image_dimensions(im[0].shape)

pix_res = dim_utils.im_physical_sizes()   # e.g. {'z': 4.0, 'y': 0.48, 'x': 0.48} µm/px
volume = np.asarray(im[0])                # [T, C, Z, Y, X]
ch_indices = [0, 1, 2]                    # confetti channels
print('volume', volume.shape, '| pix_res', pix_res)

## 2. Train the segmentation UNet (no ground-truth labels)

Supervision comes from optical-flow structure, not masks. Training input is a list of **2D**
`[T, H, W]` sequences (e.g. per-Z slices, or a chosen plane), each normalised to `[0, 1]`. Run
once and `save_model`; skip to step 3 with `load_model` on later runs.

The tunable knobs kept in `train_with_metrics` are the two-loss weights (`temporal_weight`,
`intensity_weight`) plus the optional `warp_weight`; defaults reflect the current best.

In [ ]:
# Provide your 2D training sequences, each [T, H, W] float in [0, 1]:
movies = []   # e.g. [vol_norm[:, c, z] for z in range(Z)] for one channel c

all_frames, all_metrics = coastal.prepare_data_for_unet_batch(
    movies, temporal_scales=[1, 2, 4, 8], cumulative_window=5)
tr_frames, te_frames, tr_metrics, te_metrics = coastal.train_test_split_per_movie(
    all_frames, all_metrics, train_ratio=0.8)

model, history = coastal.train_with_metrics(
    tr_frames, tr_metrics,
    num_epochs=50, embedding_dim=16,
    temporal_weight=2.0, intensity_weight=1.0, warp_weight=0.0,
    device=DEVICE)
coastal.save_model(model, 'coastal_model.pt')

In [ ]:
# Later runs: load instead of retrain.
# model = coastal.load_model('coastal_model.pt', device=DEVICE)

## 3. Segment the 3D+T volume → instance labels

`Inference3D` runs the (single-pass) `LearnedAffinityInference` on each Z-slice, then stitches
labels across Z with IOU matching (`match_masks_3d`, now true Jaccard after the audit). Output is
`instances_4d [T, Z, H, W]` — the label volume that is coastal's hand-off back to cecelia.

The kwargs below are forwarded to `LearnedAffinityInference` (note `prob_weight`, renamed from the
old `prob_merge_weight`). Tune per dataset; these are reasonable starting values.

In [ ]:
seg = coastal.Inference3D(
    model, device=DEVICE,
    seed_size=12, affinity_threshold=0.5,
    prob_weight=0.3, prob_threshold=0.3,
    embedding_blur_sigma=1.5,
    merge_affinity_threshold=0.90, merge_max_distance=1.5,
    merge_contact_brightness_threshold=0.60,
    min_component_size=10,
)

instances_4d = seg.predict_temporal_volume(
    volume, ch_indices=ch_indices,
    stitch_threshold=0.0, gap_tolerance=1, gap_iou_threshold=0.3,
    temporal_scales=[1, 2, 4], cumulative_window=2,
)
print('instances_4d', instances_4d.shape, '| labels', len(np.unique(instances_4d)) - 1)

## 4. Tracking inputs (geometric — the two cost terms that helped)

The consensus tracker uses three costs: the always-on **Mahalanobis** position gate, **`w_flow`**
(dense Farneback flow-warp), and **`w_color`** (confetti cosine). So we precompute:

- an intensity movie `[T, Z, H, W]` for optical flow (max over confetti channels here),
- `cell_flows` — per-cell measured flow, refines XY position prediction,
- `dense_flow_fields` — the dense field for the `w_flow` cost,
- `cell_intensities` — per-cell confetti intensity for the `w_color` cost.

In [ ]:
frames = volume[:, ch_indices].max(axis=1)                 # [T, Z, H, W] intensity for flow

cell_flows = coastal.compute_cell_flows(frames, instances_4d)
_feats, dense_flow_fields = coastal.compute_cell_flow_features(frames, instances_4d)
cell_intensities = coastal.extract_cell_intensities(instances_4d, volume, ch_indices)
print('cell_flows T =', len(cell_flows), '| dense_flow_fields T =', len(dense_flow_fields))

## 5. Track (Kalman + LAP; Mahalanobis + `w_flow` + `w_color`)

Tracking runs in µm (anisotropy is load-bearing — see `docs/TRACKING.md`). `w_color=1.0` is the
current honest best (improves switch-rate at ~zero continuity cost); `w_flow=0.3` helps marginally.

In [ ]:
tracks = coastal.track_sequence(
    instances_4d, pix_res,
    cell_flows=cell_flows,
    dense_flow_fields=dense_flow_fields,
    cell_intensities=cell_intensities,
    w_flow=0.3,
    w_color=1.0,
    chi2_gate=9.21, max_gap=1,
)
print('tracks:', len(tracks))

## 6. Score against confetti identity

`continuity` (↑ = less fragmentation) and `switch_rate` (↓ = fewer wrong assignments), computed
only on detected cells (no ghost-track inflation). Target: beat both at once (open problem).

In [ ]:
scores = coastal.score_tracking(
    tracks, instances_4d, volume, ch_indices, pix_res, verbose=True)
print(scores)

## 7. (Optional) tune the tracker with CMA-ES

Closed-loop search over `track_sequence`'s kept weights only (`w_flow`, `w_color`, `chi2_gate`,
`max_cost`, `process_noise`, `obs_noise`, `momentum_decay` — see `TRACKING_PARAM_BOUNDS`).
Needs the raw `volume` for colour scoring; slow, so run deliberately.

In [ ]:
# best_params, hist = coastal.optimize_tracking_cma(
#     instances_4d, volume, ch_indices, pix_res,
#     cell_flows=cell_flows, dense_flow_fields=dense_flow_fields,
#     max_evals=50)
# print(best_params)